## 第1步 监督微调（SFT）
运行环境：2 x 16GiB P100 GPU

#pip install -i https://pypi.tuna.tsinghua.edu.cn/simple deepspeed

### 定义通用的参数

In [3]:
import argparse

args = argparse.ArgumentParser(description="(Step 3) RLHF training arguments")

args.local_rank = 0
args.data_path = ["/mnt/Dahoas/rm-static"] 
args.data_split = "2,4,4" 
args.data_output_path = '/tmp/data_files'
args.max_seq_len = 256
args.num_train_epochs = 1 
args.seed = 1234 
args.per_device_train_batch_size = 1
args.per_device_eval_batch_size = 1
args.print_loss = True

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import DataLoader
from dschat.utils.data.data_utils import create_prompt_dataset

model_name_or_path = '/mnt/chinese-llama-2-7b'

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map="auto", trust_remote_code=True,
                                            offload_folder = '/tmp').eval()

train_phase = 1
train_dataset, eval_dataset = create_prompt_dataset(
    args.local_rank,
    args.data_path,
    args.data_split,
    args.data_output_path,
    train_phase,
    args.seed,
    tokenizer,
    args.max_seq_len,
    sft_only_data_path=[])

train_dataloader = DataLoader(train_dataset, batch_size=args.per_device_train_batch_size)

for epoch in range(args.num_train_epochs):
    print(
        f"Beginning of Epoch {epoch+1}/{args.num_train_epochs}, Total Micro Batches {len(train_dataloader)}")
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch, use_cache=False)
        loss = outputs.loss
        if args.print_loss:
            print(
                f"Epoch: {epoch}, Step: {step}, Rank: loss = {loss}"
            )
        loss.backward(loss)

        break;

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Beginning of Epoch 1/1, Total Micro Batches 15252
Epoch: 0, Step: 0, Rank: loss = 5.420875072479248


In [8]:
outputs.loss

tensor(5.4209, grad_fn=<ToCopyBackward0>)

In [9]:
outputs.logits

tensor([[[-1.4668, -3.6502, -0.1360,  ..., -1.3301, -0.2463, -1.1404],
         [-1.4668, -3.6468, -0.1329,  ..., -1.3304, -0.2472, -1.1407],
         [-1.4667, -3.6434, -0.1298,  ..., -1.3306, -0.2481, -1.1411],
         ...,
         [-2.7696, -1.9101, 10.7327,  ..., -0.7247, -1.9516, -1.8949],
         [-1.6405,  1.4530, 12.9067,  ..., -0.2989, -0.2526, -2.3582],
         [-1.0339,  5.5531, 15.5704,  ...,  1.9166,  0.3002,  0.5470]]],
       grad_fn=<ToCopyBackward0>)

In [10]:
outputs.logits.size()   #[bacth, seq_length, vocab_size]

torch.Size([1, 256, 55296])